# 01 - Data preparation and first look

Start by checking the raw Olympic results and country mapping. The goal is to make one row per country per Summer Games, then add World Bank indicators. Do not train a model until the country matching and medal counting rules are understood.

Expected raw files in `data/raw/`: `athlete_events.csv`, `noc_regions.csv`, and a World Bank indicators CSV. The first two are from Kaggle's *120 years of Olympic history: athletes and results*.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT.resolve()))
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print('Project folder:', PROJECT_ROOT.name)
print('Raw data folder: data/raw')

Project folder: olympic-medal-prediction
Raw data folder: data/raw


## 1. Check which source files are present

Download the files listed in the README and put them in `data/raw/`. Keep the original filenames where possible.

In [2]:
for filename in ['athlete_events.csv', 'noc_regions.csv']:
    path = RAW_DIR / filename
    print(f"{'FOUND' if path.exists() else 'MISSING'}: data/raw/{filename}")

print('Raw folder contents:', [p.name for p in RAW_DIR.iterdir() if p.is_file()])

FOUND: data/raw/athlete_events.csv
FOUND: data/raw/noc_regions.csv
Raw folder contents: ['.gitkeep', '7d88ef73-827c-4b35-a13b-904df2cbae3d_Data.csv', '7d88ef73-827c-4b35-a13b-904df2cbae3d_Series - Metadata.csv', 'athlete_events.csv', 'noc_regions.csv', 'tokyo_2020_medals_total.csv']


## 2. Load and inspect the Olympic files

Run this after adding both Olympic CSVs. Check the available columns and country codes before deciding how to aggregate team events. In the athlete-level source, medal entries for a team event may repeat for every athlete on that team.

In [3]:
athletes_path = RAW_DIR / 'athlete_events.csv'
noc_path = RAW_DIR / 'noc_regions.csv'

if athletes_path.exists() and noc_path.exists():
    athletes = pd.read_csv(athletes_path)
    noc_regions = pd.read_csv(noc_path)
    display(athletes.head())
    print('Olympic rows/columns:', athletes.shape)
    print('Olympic columns:', athletes.columns.tolist())
    display(noc_regions.head())
    print('Country mapping rows/columns:', noc_regions.shape)
else:
    print('Add athlete_events.csv and noc_regions.csv to data/raw/, then rerun this cell.')

,ID,Name,Sex,Age,Height,Weight,Team,NOC,Games,Year,Season,City,Sport,Event,Medal
0,1,A Dijiang,M,24.0,180.0,80.0,China,CHN,1992 Summer,1992,Summer,Barcelona,Basketball,Basketball Men's Basketball,NaN
1,2,A Lamusi,M,23.0,170.0,60.0,China,CHN,2012 Summer,2012,Summer,London,Judo,Judo Men's Extra-Lightweight,NaN
2,3,Gunnar Nielsen Aaby,M,24.0,NaN,NaN,Denmark,DEN,1920 Summer,1920,Summer,Antwerpen,Football,Football Men's Football,NaN
3,4,Edgar Lindenau Aabye,M,34.0,NaN,NaN,Denmark/Sweden,DEN,1900 Summer,1900,Summer,Paris,Tug-Of-War,Tug-Of-War Men's Tug-Of-War,Gold
4,5,Christine Jacoba Aaftink,F,21.0,185.0,82.0,Netherlands,NED,1988 Winter,1988,Winter,Calgary,Speed Skating,Speed Skating Women's 500 metres,NaN


Olympic rows/columns: (271116, 15)
Olympic columns: ['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games', 'Year', 'Season', 'City', 'Sport', 'Event', 'Medal']


,NOC,region,notes
0,AFG,Afghanistan,NaN
1,AHO,Curacao,Netherlands Antilles
2,ALB,Albania,NaN
3,ALG,Algeria,NaN
4,AND,Andorra,NaN


Country mapping rows/columns: (230, 3)


## 3. Limit to Summer Games and inspect coverage

The assignment focuses on Summer Olympics. Verify that each Olympic year and medal category looks sensible before building the country-year target.

In [4]:
if 'athletes' in globals():
    summer = athletes.loc[athletes['Season'].eq('Summer')].copy()
    print('Summer Games years:', sorted(summer['Year'].dropna().unique().tolist()))
    print('Missing values in selected fields:')
    display(summer[['Year', 'NOC', 'Team', 'Sport', 'Event', 'Medal']].isna().sum())
    print('Medal labels:', summer['Medal'].value_counts(dropna=False).to_dict())
else:
    print('Load the Olympic CSVs in the previous section first.')

Summer Games years: [1896, 1900, 1904, 1906, 1908, 1912, 1920, 1924, 1928, 1932, 1936, 1948, 1952, 1956, 1960, 1964, 1968, 1972, 1976, 1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008, 2012, 2016]
Missing values in selected fields:


Year          0
NOC           0
Team          0
Sport         0
Event         0
Medal    188464
dtype: int64

Medal labels: {nan: 188464, 'Gold': 11459, 'Bronze': 11409, 'Silver': 11220}


## 4. Next decisions

Before aggregating, record answers to these questions in your notes:

- What identifies a unique medal-winning event result in these files?
- How will team-event athlete duplicates be removed without losing legitimate medals?
- Which NOC codes or historical country names need to be mapped or excluded?
- Which Olympic years will be used for training and for the time-based holdout?
- For a retrospective 2020 forecast, which socioeconomic data year was available before the Games?

After resolving those choices, create the country-year medal table, merge World Bank features by country code and year, and save the result under `data/processed/`.

In [5]:
# Exclude 1906, then keep only rows that represent medals.
summer_official = summer.loc[summer["Year"].ne(1906)].drop_duplicates()
medal_rows = summer_official.loc[summer_official["Medal"].notna()].copy()

# Team medals are repeated for each athlete. Count each NOC's medal
# once per event and medal type.
event_medals = medal_rows.drop_duplicates(
    subset=["Year", "NOC", "Sport", "Event", "Medal"]
)

# Count Gold, Silver, Bronze for each NOC at each Games.
country_year = (
    event_medals.groupby(["Year", "NOC", "Medal"])
    .size()
    .unstack(fill_value=0)
)

for medal in ["Gold", "Silver", "Bronze"]:
    if medal not in country_year.columns:
        country_year[medal] = 0

country_year["Total"] = country_year[["Gold", "Silver", "Bronze"]].sum(axis=1)
country_year = country_year.reset_index()

# Add the readable country/region name from the NOC mapping.
country_year = country_year.merge(
    noc_regions[["NOC", "region"]].drop_duplicates("NOC"),
    on="NOC",
    how="left"
)

print("Country-Games rows:", len(country_year))
print("Unmapped NOCs:", sorted(country_year.loc[country_year["region"].isna(), "NOC"].unique()))
display(country_year.sort_values(["Year", "Total"], ascending=[True, False]).head(20))

Country-Games rows: 1257
Unmapped NOCs: ['SGP']


,Year,NOC,Bronze,Gold,Silver,Total,region
6,1896,GRE,17,10,17,44,Greece
9,1896,USA,2,11,6,19,USA
5,1896,GER,2,7,5,14,Germany
3,1896,FRA,2,5,4,11,France
4,1896,GBR,3,3,3,9,UK
2,1896,DEN,3,1,2,6,Denmark
7,1896,HUN,3,2,1,6,Hungary
1,1896,AUT,2,2,1,5,Austria
0,1896,AUS,1,2,0,3,Australia
8,1896,SUI,0,1,2,3,Switzerland


In [6]:
# Add every NOC that participated, including those with no medals.
participation = summer_official[["Year", "NOC"]].drop_duplicates()
medal_counts = country_year.drop(columns="region")

country_year_full = (
    participation
    .merge(medal_counts, on=["Year", "NOC"], how="left", validate="one_to_one")
    .merge(
        noc_regions[["NOC", "region"]].drop_duplicates("NOC"),
        on="NOC",
        how="left",
        validate="many_to_one"
    )
)

for medal in ["Gold", "Silver", "Bronze", "Total"]:
    country_year_full[medal] = country_year_full[medal].fillna(0).astype(int)

print("All participating country-Games rows:", len(country_year_full))
print("Zero-medal rows:", (country_year_full["Total"] == 0).sum())
print("Unmapped NOCs:", sorted(
    country_year_full.loc[country_year_full["region"].isna(), "NOC"].unique()
))
display(country_year_full.head(20))

All participating country-Games rows: 2789
Zero-medal rows: 1532
Unmapped NOCs: ['ROT', 'SGP', 'TUV', 'UNK']


,Year,NOC,Bronze,Gold,Silver,Total,region
0,1992,CHN,15,16,22,53,China
1,2012,CHN,23,38,27,88,China
2,1920,DEN,1,3,9,13,Denmark
3,1900,DEN,2,2,3,7,Denmark
4,1932,NED,1,2,5,8,Netherlands
5,1952,FIN,13,6,3,22,Finland
6,2000,FIN,1,2,1,4,Finland
7,1996,FIN,1,1,2,4,Finland
8,1912,FIN,9,9,8,26,Finland
9,1920,FIN,9,15,10,34,Finland


In [7]:
# Add known geographic mappings; keep non-country teams/placeholders out of the country model.
region_lookup = (
    noc_regions.drop_duplicates("NOC")
    .set_index("NOC")["region"]
    .to_dict()
)
region_lookup.update({"SGP": "Singapore", "TUV": "Tuvalu"})
country_year_full["region"] = country_year_full["NOC"].map(region_lookup)

from src.data_preparation import NON_COUNTRY_NOCS

# Historical composite teams and placeholder NOCs are excluded from the country-level model.
country_year_model = country_year_full.loc[
    ~country_year_full["NOC"].isin(NON_COUNTRY_NOCS)
].copy()

print("Country-Games rows retained:", len(country_year_model))
print("Excluded non-country NOC rows:", len(country_year_full) - len(country_year_model))
print("Still-unmapped NOCs:", sorted(
    country_year_model.loc[country_year_model["region"].isna(), "NOC"].unique()
))
display(country_year_model.loc[country_year_model["NOC"].isin(["SGP", "TUV"]), ["Year", "NOC", "region", "Gold", "Silver", "Bronze", "Total"]].head())

Country-Games rows retained: 2710
Excluded non-country NOC rows: 79
Still-unmapped NOCs: []


,Year,NOC,region,Gold,Silver,Bronze,Total
218,1992,SGP,Singapore,0,0,0,0
219,1996,SGP,Singapore,0,0,0,0
895,2016,SGP,Singapore,1,0,0,1
1432,1984,SGP,Singapore,0,0,0,0
1433,1988,SGP,Singapore,0,0,0,0


## 5. Merge pre-Games World Bank indicators

The preparation script reshapes the DataBank export, applies the NOC-to-World Bank crosswalk, and joins GDP, population, and land area from the year before each Games. It leaves missing indicators blank and writes the prepared table and crosswalk under `data/processed/`.

In [8]:
from src.data_preparation import build_modeling_table

modeling_table, modeling_diagnostics = build_modeling_table(RAW_DIR, PROCESSED_DIR)
path_labels = {'Output file', 'World Bank country-year features file', 'NOC crosswalk file'}
for label, value in modeling_diagnostics.items():
    if label in path_labels:
        value = Path(value).relative_to(PROJECT_ROOT).as_posix()
    print(f"{label}: {value}")

display(modeling_table.head())

Olympic country-Games rows before feature matching: 2789
World Bank indicator rows after reshaping: 17490
Olympic rows with a World Bank prior-year available: 2180
Olympic rows with a World Bank country mapping and prior-year record: 2119
Unmapped or non-country NOCs: [{'NOC': 'AHO', 'Olympic_Region': 'Curacao', 'Match_Type': 'unmatched'}, {'NOC': 'COK', 'Olympic_Region': 'Cook Islands', 'Match_Type': 'unmatched'}, {'NOC': 'EUN', 'Olympic_Region': 'Russia', 'Match_Type': 'unmatched'}, {'NOC': 'FRG', 'Olympic_Region': 'Germany', 'Match_Type': 'unmatched'}, {'NOC': 'GDR', 'Olympic_Region': 'Germany', 'Match_Type': 'unmatched'}, {'NOC': 'IOA', 'Olympic_Region': 'Individual Olympic Athletes', 'Match_Type': 'unmatched'}, {'NOC': 'ROT', 'Olympic_Region': nan, 'Match_Type': 'unmatched'}, {'NOC': 'SCG', 'Olympic_Region': 'Serbia', 'Match_Type': 'unmatched'}, {'NOC': 'TCH', 'Olympic_Region': 'Czech Republic', 'Match_Type': 'unmatched'}, {'NOC': 'URS', 'Olympic_Region': 'Russia', 'Match_Type': '

,Year,NOC,Bronze,Gold,Silver,Total,Athlete_Count,Olympic_Region,WB_Country_Code,WB_Country_Name,Match_Type,Feature_Year,Land_Area_Sq_Km,GDP_USD,Population
0,1992,CHN,15,16,22,53,243,China,CHN,China,NOC region matches World Bank country name,1991,9388250.0,3.845105e+11,1.150780e+09
1,2012,CHN,23,38,27,88,365,China,CHN,China,NOC region matches World Bank country name,2011,9388211.0,7.671757e+12,1.345035e+09
2,2000,FIN,1,2,1,4,70,Finland,FIN,Finland,NOC region matches World Bank country name,1999,304590.0,1.352342e+11,5.165474e+06
3,1996,FIN,1,1,2,4,76,Finland,FIN,Finland,NOC region matches World Bank country name,1995,304590.0,1.340605e+11,5.107790e+06
4,2008,NOR,1,3,5,9,84,Norway,NOR,Norway,NOC region matches World Bank country name,2007,365244.0,4.078138e+11,4.709153e+06
